In [3]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from gluonts.dataset.repository.datasets import get_dataset, dataset_recipes
from gluonts.dataset.util import to_pandas

import torch
print(torch.__version__)

2.10.0+cu126


# Load Datasets from Gluonts

In [3]:
print(f"Available datasets: {list(dataset_recipes.keys())}")

Available datasets: ['constant', 'exchange_rate', 'solar-energy', 'electricity', 'traffic', 'exchange_rate_nips', 'electricity_nips', 'traffic_nips', 'solar_nips', 'wiki2000_nips', 'wiki-rolling_nips', 'taxi_30min', 'kaggle_web_traffic_with_missing', 'kaggle_web_traffic_without_missing', 'kaggle_web_traffic_weekly', 'm1_yearly', 'm1_quarterly', 'm1_monthly', 'nn5_daily_with_missing', 'nn5_daily_without_missing', 'nn5_weekly', 'tourism_monthly', 'tourism_quarterly', 'tourism_yearly', 'cif_2016', 'london_smart_meters_without_missing', 'wind_farms_without_missing', 'car_parts_without_missing', 'dominick', 'fred_md', 'pedestrian_counts', 'hospital', 'covid_deaths', 'kdd_cup_2018_without_missing', 'weather', 'm3_monthly', 'm3_quarterly', 'm3_yearly', 'm3_other', 'm4_hourly', 'm4_daily', 'm4_weekly', 'm4_monthly', 'm4_quarterly', 'm4_yearly', 'm5', 'uber_tlc_daily', 'uber_tlc_hourly', 'airpassengers', 'australian_electricity_demand', 'electricity_hourly', 'electricity_weekly', 'rideshare_wit

In [ ]:
d_name = "exchange_rate"
dataset = get_dataset(d_name, regenerate=False)
dataset.metadata

In [ ]:
# to TimeSeriesDataSet
train_iter = iter(dataset.train)
test_iter = iter(dataset.test)

data_df = pd.DataFrame(columns=['datetime', 'sensor', 'value'])
for i in range(int(dataset.metadata.feat_static_cat[0].cardinality)):
    train_entry = next(train_iter)
    test_entry = next(test_iter)

    train_series = to_pandas(train_entry)
    test_series = to_pandas(test_entry)

    sensor_readings = pd.concat([train_series, test_series[train_series.index[-1]+1:]]).to_frame(name='value')
    sensor_readings.reset_index(inplace=True, names=['datetime'])
    sensor_readings['sensor'] = i

    data_df = pd.concat([data_df, sensor_readings])

data_df = data_df.astype(dict(datetime='datetime64[ns]', sensor=str))

time_idx_df = pd.DataFrame(data_df['datetime'].unique(), columns=["datetime"]).sort_values(by="datetime").reset_index(drop=True).reset_index(names="time_idx")
data_df = pd.merge(data_df, time_idx_df, left_on="datetime", right_on="datetime", how="left")

data_df.to_csv("../datasets/%s.csv"%(d_name), index=False)
print(data_df.shape[0])
print(data_df.head())

In [5]:
!pip install tables

  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 36.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 32.7 MB/s  0:00:00
Using cached py_cpuinfo-9.0.0-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [tables]2m3/4 [tables]


(15123, 207)


# Load Datasets from H5

In [67]:
d_name = "pems03_flow"  # pems-bay, metr-la, pemsd7m, gz-metro, hz-metro, pems03_flow, pems04_flow, pems07_flow, pems08_flow, seattle
data_df = pd.read_hdf("../datasets/%s.h5"%(d_name))
print(data_df.shape)

(26208, 358)


In [68]:


data_df.reset_index(inplace=True)
data_df['time_idx'] = np.arange(data_df.shape[0])
data_df.rename(columns={'index':'datetime'}, inplace=True)

data_df = pd.melt(data_df, id_vars=[data_df.columns[0], data_df.columns[-1]], var_name='sensor', value_vars=data_df.columns[1:-1])
data_df = data_df.astype(dict(sensor=int, time_idx=int))
data_df = data_df.astype(dict(sensor=str))
print(data_df.shape)
print(data_df.head())

data_df.to_csv("../datasets/%s.csv"%("pems03_flow"), index=False)



(9382464, 4)
   datetime  time_idx sensor  value
0         0         0      0   20.0
1         1         1      0   22.0
2         2         2      0   22.0
3         3         3      0   50.0
4         4         4      0   37.0


In [60]:
data_df

,datetime,0,1,2,3,4,5,6,7,8,...,349,350,351,352,353,354,355,356,357,time_idx
0,0,20.0,20.0,182.0,182.0,91.0,182.0,136.0,91.0,91.0,...,63.0,63.0,63.0,125.0,114.0,63.0,63.0,115.0,63.0,0
1,1,22.0,22.0,174.0,174.0,87.0,174.0,131.0,87.0,87.0,...,62.0,62.0,62.0,144.0,109.0,62.0,63.0,109.0,62.0,1
2,2,22.0,22.0,183.0,183.0,92.0,183.0,139.0,92.0,92.0,...,57.0,57.0,57.0,131.0,116.0,57.0,57.0,115.0,57.0,2
3,3,50.0,49.0,137.0,139.0,60.0,158.0,111.0,57.0,55.0,...,109.0,30.0,58.0,106.0,102.0,52.0,111.0,88.0,48.0,3
4,4,37.0,35.0,128.0,123.0,54.0,131.0,111.0,46.0,52.0,...,77.0,31.0,55.0,109.0,127.0,49.0,109.0,76.0,47.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26203,26203,54.0,53.0,120.0,128.0,66.0,149.0,110.0,56.0,61.0,...,63.0,35.0,67.0,51.0,147.0,59.0,134.0,144.0,48.0,26203
26204,26204,52.0,55.0,109.0,107.0,58.0,126.0,114.0,51.0,55.0,...,65.0,36.0,72.0,61.0,143.0,64.0,139.0,130.0,61.0,26204
26205,26205,43.0,39.0,144.0,143.0,55.0,147.0,99.0,62.0,61.0,...,89.0,32.0,55.0,61.0,132.0,57.0,124.0,104.0,47.0,26205
26206,26206,27.0,27.0,115.0,114.0,62.0,129.0,91.0,41.0,47.0,...,50.0,35.0,61.0,72.0,132.0,56.0,123.0,112.0,39.0,26206


In [14]:
def load_brussels(data_name=None):
    """
    Load data from CSV and return np.ndarray of shape (T, 145).

    test set will be defined as last 10% of time steps, outside of this function.
    """
    data = pd.read_csv(
        "/home/seyed/PycharmProjects/step/STEP/datasets/Bru/merged_data.csv",
        index_col=0,
        parse_dates=[0],
    )
    data = (
        data.interpolate(method="spline", order=3, limit_area="inside")
            .ffill()
            .bfill()
    )
    # data.values is (T, 145)
    return data


data_df = load_brussels()
print(data_df.shape)

(15123, 207)


In [8]:
df_2024 = pd.read_csv("/home/seyed/Documents/BM_data_2024/process/debug_final_pivot_data_2023-12-30_2079-04-24.csv", parse_dates=["start_timestamp"])

In [10]:
df_2024 = df_2024[(df_2024.start_timestamp > pd.Timestamp("2024-01-01 00:00")) & (df_2024.start_timestamp < pd.Timestamp("2024-01-31 00:00"))]

In [12]:
df_2024.to_csv("/home/seyed/forked/T3Time/scripts/brussels.csv", index=True)

In [15]:
# detect columns with ratio of zero values > 0.5 and drop them
zero_ratio = (data_df == 0).mean(axis=0)
prune_data_df = data_df.loc[:, zero_ratio <= 0.5]

print(f"Remaining columns: {prune_data_df.shape[1]}")

Remaining columns: 195


In [17]:
data_df.to_csv("/home/seyed/forked/T3Time/scripts/brussels.csv", index=True)

In [18]:
data_df.head()

,BXLAND031448F1,BXLAND031450F1,BXLAND031451F1,BXLAND034219F1,BXLAND034220F1,BXLAND034221F1,BXLAND034222F1,BXLAND034224F1,BXLAND034225F1,BXLAND034232F1,...,BXLKOE034178F1,BXLKOE034179F1,BXLKOE037164B1,BXLKOE037165B1,BXLKOE037167B1,BXLKOE126857B1,BXLKOE126918F1,BXLKOE126923F1,BXLKOE126924F1,BXLKOE126925F1
start_timestamp,,,,,,,,,,,,,,,,,,,,,
2023-11-19 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-11-19 00:05:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-11-19 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-11-19 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-11-19 00:20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [71]:
d_name = "brussels"

In [154]:
filtered_columns = data_df.loc[:, zero_ratio <= 0.5].columns

In [79]:
prune_data_df.drop(["start_timestamp"], axis=1, inplace=True)

/tmp/ipykernel_2992361/3542680190.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prune_data_df.drop(["start_timestamp"], axis=1, inplace=True)


In [80]:
prune_data_df.columns[-2], prune_data_df.columns[-1]

('time_idx', 'datetime')

In [83]:
prune_data_df.columns[0:-2]

Index(['BXLAND031448F1', 'BXLAND031450F1', 'BXLAND031451F1', 'BXLAND034219F1',
       'BXLAND034220F1', 'BXLAND034221F1', 'BXLAND034222F1', 'BXLAND034224F1',
       'BXLAND034225F1', 'BXLAND034232F1',
       ...
       'BXLKOE034178F1', 'BXLKOE034179F1', 'BXLKOE037164B1', 'BXLKOE037165B1',
       'BXLKOE037167B1', 'BXLKOE126857B1', 'BXLKOE126918F1', 'BXLKOE126923F1',
       'BXLKOE126924F1', 'BXLKOE126925F1'],
      dtype='object', length=195)

In [65]:
d_name = "brussels"

In [84]:
# prune_data_df.reset_index(inplace=True)
# prune_data_df['time_idx'] = np.arange(prune_data_df.shape[0])
# prune_data_df.rename(columns={'index':'datetime'}, inplace=True)
# prune_data_df['datetime'] = pd.date_range(start='2023-11-19 00:00:00', periods=len(prune_data_df['start_timestamp']), freq='5min')
prune_data_df = pd.melt(prune_data_df, id_vars=[prune_data_df.columns[-2], prune_data_df.columns[-1]], var_name='sensor', value_vars=prune_data_df.columns[0:-2])
prune_data_df = prune_data_df.astype(dict(sensor=str, time_idx=int))
prune_data_df = prune_data_df.astype(dict(sensor=str))
print(prune_data_df.shape)
print(prune_data_df.head())

prune_data_df.to_csv("../datasets/%s.csv"%(d_name), index=False)

(2948985, 4)
   time_idx            datetime          sensor  value
0         0 2023-11-19 00:00:00  BXLAND031448F1    0.0
1         1 2023-11-19 00:05:00  BXLAND031448F1    0.0
2         2 2023-11-19 00:10:00  BXLAND031448F1    0.0
3         3 2023-11-19 00:15:00  BXLAND031448F1    0.0
4         4 2023-11-19 00:20:00  BXLAND031448F1    0.0


import pickle

In [1]:
import json

with open('/media/seyed/D2DA77DADA77B975/BM/data/1/18/raw_data.txt', 'r') as f:
    raw_data = json.load(f)

JSONDecodeError: Extra data: line 3396 column 2 (char 110963)

In [6]:
pd.json_normalize(raw_data)

,_aggregation_id,_creation_timestamp,_end_timestamp,_expiration,_start_timestamp,_type,_update_timestamp,count,period,subtype,...,speed_classes.[30;40[,speed_classes.[40;50[,speed_classes.[50;60[,speed_classes.[0;10[,speed_classes.[60;70[,speed_classes.[70;80[,speed_classes.[80;90[,speed_classes.[90;100[,speed_classes.[110;120[,speed_classes.[100;110[
0,1700465700000#PT5M#lightVehicle#null#5f883210a...,2023-11-20 07:39:14.246000,2023-11-20 07:35:00,2023-11-25 07:39:11.228000,2023-11-20 07:30:00,counting,2023-11-20 07:39:14.246000,1,PT5M,counting,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1700465700000#PT5M#lightVehicle#null#5f883629a...,2023-11-20 07:39:14.246000,2023-11-20 07:35:00,2023-11-25 07:39:11.233000,2023-11-20 07:30:00,counting,2023-11-20 07:39:14.246000,1,PT5M,counting,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1700465700000#PT5M#lightVehicle#BE#5a78690b48d...,2023-11-20 07:39:14.246000,2023-11-20 07:35:00,2023-11-25 07:39:11.242000,2023-11-20 07:30:00,counting,2023-11-20 07:39:14.246000,45,PT5M,counting,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1700465700000#PT5M#lightVehicle#BE#5a33df4d53e...,2023-11-20 07:39:14.246000,2023-11-20 07:35:00,2023-11-25 07:39:11.234000,2023-11-20 07:30:00,counting,2023-11-20 07:39:14.246000,52,PT5M,counting,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1700465700000#PT5M#lightVehicle#BE#5f02b5cb73b...,2023-11-20 07:39:14.246000,2023-11-20 07:35:00,2023-11-25 07:39:11.245000,2023-11-20 07:30:00,counting,2023-11-20 07:39:14.246000,70,PT5M,counting,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160078,1700520600000#PT5M#lightVehicle#5f9132ce6905da...,2023-11-20 22:54:46.191000,2023-11-20 22:50:00,2023-11-25 22:54:46.168000,2023-11-20 22:45:00,NaN,2023-11-20 22:54:46.191000,1,PT5M,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
160079,1700520600000#PT5M#lightVehicle#6023eda249afa4...,2023-11-20 22:54:46.191000,2023-11-20 22:50:00,2023-11-25 22:54:46.168000,2023-11-20 22:45:00,NaN,2023-11-20 22:54:46.191000,25,PT5M,NaN,...,NaN,19.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
160080,1700520600000#PT5M#lightVehicle#5f7d875ead2d18...,2023-11-20 22:54:46.191000,2023-11-20 22:50:00,2023-11-25 22:54:46.168000,2023-11-20 22:45:00,NaN,2023-11-20 22:54:46.191000,6,PT5M,NaN,...,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
160081,1700520600000#PT5M#lightVehicle#5f9135cb6905da...,2023-11-20 22:54:46.191000,2023-11-20 22:50:00,2023-11-25 22:54:46.168000,2023-11-20 22:45:00,NaN,2023-11-20 22:54:46.191000,4,PT5M,NaN,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
dfs = []
for key in raw_data:
    dfs.append(pd.json_normalize(key))
dfs = pd.concat(dfs)

In [8]:
ddd = dfs[dfs["subtype"] == "counting"]

In [9]:
ddd.columns

Index(['_aggregation_id', '_creation_timestamp', '_end_timestamp',
       '_expiration', '_start_timestamp', '_type', '_update_timestamp',
       'count', 'period', 'subtype', 'type', 'vehicle_type',
       'location.coordinates', 'location.type', 'source.name.fr',
       'source.name.vi', 'source.name.nl', 'source.name.de',
       'source.name.nl-BE', 'source.name.en', 'source.id', 'source._id',
       'nationality', 'source.name.fr-BE', 'measures', 'median_duration',
       'median_speed', 'resources.name.fr', 'resources.name.vi',
       'resources.name.nl', 'resources.name.de', 'resources.name.nl-BE',
       'resources.name.en', 'resources.id', 'resources._id',
       'speed_classes.[20;30[', 'speed_classes.[10;20[',
       'speed_classes.[30;40[', 'speed_classes.[40;50[',
       'speed_classes.[50;60[', 'speed_classes.[0;10[',
       'speed_classes.[60;70[', 'speed_classes.[70;80[',
       'speed_classes.[80;90[', 'speed_classes.[90;100[',
       'speed_classes.[110;120[', 'speed_c

In [10]:
locations = {}

for key in ddd["source.id"].drop_duplicates():
    locations[key] = ddd.loc[ddd["source.id"] == key]["location.coordinates"].iloc[0]

In [11]:
locations

{'BXLAND135656B1': [4.324086, 50.844197],
 'BXLAND135417B1': [4.325761, 50.845194],
 'BXLMOL034189F1': [4.334110920203833, 50.856668996612875],
 'BXLBXL125638B1': [4.388069172608651, 50.84146550625018],
 'BXLBXL031438F1': [4.326966, 50.898064],
 'BXLFOR034193F1': [4.30547721101096, 50.801439162695424],
 'BXLIXL127916F1': [4.353909486966835, 50.82260947052188],
 'BXLUCC034677F1': [4.335111107242816, 50.77474764098082],
 'BXLWSP034669F1': [4.465217682070772, 50.835227877859985],
 'BXLSCH035888F1': [4.398806, 50.85517],
 'BXLEVE128267B1': [4.402418653845467, 50.865947845015896],
 'BXLUCC034162F1': [4.348713194870293, 50.80380801132337],
 'BXLWSL134917F1': [4.445405006408691, 50.85540008544922],
 'BXLAND135516F1': [4.291268, 50.817075734445154],
 'BXLBXL034148F1': [4.341321270109129, 50.85304966482994],
 'BXLAUD134958F1': [4.43453636441803, 50.8133750525088],
 'BXLWSL128296F1': [4.432437509705303, 50.860020199212215],
 'BXLETT034023F1': [4.393060207366943, 50.82422637939453],
 'BXLMOL13541

In [12]:
import pickle

In [13]:
pickle.dump(locations, open("../datasets/locations.pkl", "wb")) 

In [140]:
def get_adjacency_matrix(distance_df, sensor_ids, normalized_k=0.1):
    """

    :param distance_df: data frame with three columns: [from, to, distance].
    :param sensor_ids: list of sensor ids.
    :param normalized_k: entries that become lower than normalized_k after normalization are set to zero for sparsity.
    :return:
    """
    num_sensors = len(sensor_ids)
    dist_mx = np.zeros((num_sensors, num_sensors), dtype=np.float32)
    dist_mx[:] = np.inf
    # Builds sensor id to index map.
    sensor_id_to_ind = {}
    for i, sensor_id in enumerate(sensor_ids):
        sensor_id_to_ind[sensor_id] = i

    # Fills cells in the matrix with distances.
    for row in distance_df.values:
        if row[0] not in sensor_id_to_ind or row[1] not in sensor_id_to_ind:
            continue
        dist_mx[sensor_id_to_ind[row[0]], sensor_id_to_ind[row[1]]] = row[2]

    # Calculates the standard deviation as theta.
    distances = dist_mx[~np.isinf(dist_mx)].flatten()
    std = distances.std()
    adj_mx = np.exp(-np.square(dist_mx / std))
    # Make the adjacent matrix symmetric by taking the max.
    # adj_mx = np.maximum.reduce([adj_mx, adj_mx.T])

    # Sets entries that lower than a threshold, i.e., k, to zero for sparsity.
    adj_mx[adj_mx < normalized_k] = 0
    return sensor_ids, sensor_id_to_ind, adj_mx



In [142]:
def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance in kilometers between two points
    on the Earth specified in decimal degrees.
    """
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of Earth in kilometers. Use 3956 for miles.
    return c * r

def compute_distance_df(locations: dict):
    """
    return a data frame with three columns: [from, to, distance] for all pairs of sensors.
    """
    sensor_ids = list(locations.keys())
    distance_data = []
    for i in range(len(sensor_ids)):
        for j in range(i + 1, len(sensor_ids)):
            from_id = sensor_ids[i]
            to_id = sensor_ids[j]
            lon1, lat1 = locations[from_id]
            lon2, lat2 = locations[to_id]
            distance = haversine(lon1, lat1, lon2, lat2)
            distance_data.append([from_id, to_id, distance])
            distance_data.append([to_id, from_id, distance])  # Add the reverse direction

    distance_df = pd.DataFrame(distance_data, columns=['from', 'to', 'distance'])
    return distance_df

In [143]:
distance_df = compute_distance_df(locations)

In [147]:
distance_df[distance_df["distance"]<5]

,from,to,distance
0,BXLAND135656B1,BXLAND135417B1,0.161620
1,BXLAND135417B1,BXLAND135656B1,0.161620
2,BXLAND135656B1,BXLMOL034189F1,1.555177
3,BXLMOL034189F1,BXLAND135656B1,1.555177
4,BXLAND135656B1,BXLBXL125638B1,4.502771
...,...,...,...
127793,BXLSTJ125837F1,BXLIXL135837F1,3.613513
127794,BXLWAT031457F1,BXLWAT031458F1,0.025756
127795,BXLWAT031458F1,BXLWAT031457F1,0.025756
127804,BXLSCH135879F1,BXLSTJ125837F1,1.123575


In [156]:
filtered_columns.tolist()

['BXLAND031448F1',
 'BXLAND031450F1',
 'BXLAND031451F1',
 'BXLAND034219F1',
 'BXLAND034220F1',
 'BXLAND034221F1',
 'BXLAND034222F1',
 'BXLAND034224F1',
 'BXLAND034225F1',
 'BXLAND034232F1',
 'BXLAND034237B1',
 'BXLAND034238B1',
 'BXLAND034239F1',
 'BXLAND034240F1',
 'BXLAND034244F1',
 'BXLAND034685F1',
 'BXLAND034686F1',
 'BXLAND036505F1',
 'BXLAND039685F1',
 'BXLAND131286F1',
 'BXLAND131299F1',
 'BXLAND134439F1',
 'BXLAND134656F1',
 'BXLAND134657F1',
 'BXLAND134658F1',
 'BXLAND135417B1',
 'BXLAND135418B1',
 'BXLAND135419B1',
 'BXLAND135656B1',
 'BXLAND135658B1',
 'BXLAND135659B1',
 'BXLAND135936F1',
 'BXLAUD031459F1',
 'BXLAUD034150F1',
 'BXLAUD036804F1',
 'BXLAUD127156F1',
 'BXLAUD133137B1',
 'BXLAUD133166F1',
 'BXLAUD134956F1',
 'BXLAUD134958F1',
 'BXLAUD138737F1',
 'BXLBSA034173F1',
 'BXLBSA034174F1',
 'BXLBSA034175F1',
 'BXLBSA034176F1',
 'BXLBSA034184F1',
 'BXLBSA126899F1',
 'BXLBSA126936F1',
 'BXLBSA131672F1',
 'BXLBSA137640B1',
 'BXLBSA137641B1',
 'BXLBSA137642B1',
 'BXLBSA1376

In [159]:
filtered_distance = distance_df[distance_df["from"].isin(filtered_columns) & distance_df["to"].isin(filtered_columns)]

In [160]:
filtered_columns

Index(['BXLAND031448F1', 'BXLAND031450F1', 'BXLAND031451F1', 'BXLAND034219F1',
       'BXLAND034220F1', 'BXLAND034221F1', 'BXLAND034222F1', 'BXLAND034224F1',
       'BXLAND034225F1', 'BXLAND034232F1',
       ...
       'BXLKOE034178F1', 'BXLKOE034179F1', 'BXLKOE037164B1', 'BXLKOE037165B1',
       'BXLKOE037167B1', 'BXLKOE126857B1', 'BXLKOE126918F1', 'BXLKOE126923F1',
       'BXLKOE126924F1', 'BXLKOE126925F1'],
      dtype='object', length=195)

In [161]:
adj = get_adjacency_matrix(filtered_distance, filtered_columns.tolist(), normalized_k=0.1)

In [164]:
import pickle

In [165]:
pickle.dump(adj, open("adjacency_matrix.pkl", "wb"))

In [106]:
fff = pickle.load(open('/home/seyed/PycharmProjects/step/STEP/datasets/Bru/columns_index_data_dataset.pkl', 'rb'))

In [108]:
fff.head()

0    BXLAND031448F1
1    BXLAND031450F1
2    BXLAND031451F1
3    BXLAND034219F1
4    BXLAND034220F1
dtype: object

In [104]:
with open('../datasets/pred_horizon_dict.pkl', 'rb') as f:
    pred_horizon_dict = pickle.load(f)
with open('../datasets/pred_rolling_dict_v1.pkl', 'rb') as f:
    pred_rolling_dict = pickle.load(f)
with open('../datasets/dataset_freq_v1.pkl', 'rb') as f:
    dataset_freq_dict = pickle.load(f)

In [105]:
pred_rolling_dict["brussels"]

24

In [97]:
dataset_freq_dict["brussels"] = "5min"

In [98]:
pred_horizon_dict["brussels"], pred_rolling_dict["brussels"] = pred_horizon_dict["pems03_flow"], pred_rolling_dict["pems03_flow"]

In [99]:
pickle.dump(pred_horizon_dict, open('../datasets/pred_horizon_dict_v1.pkl', 'wb'))
pickle.dump(pred_rolling_dict, open('../datasets/pred_rolling_dict_v1.pkl', 'wb'))
pickle.dump(dataset_freq_dict, open('../datasets/dataset_freq_v1.pkl', 'wb'))